In [4]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils(
    "Structured Streaming with Files",
    master_url="spark://spark-master:7077"
)

su.spark

In [8]:
import sys
sys.path.insert(0, "/opt/spark/work-dir/src/producers")

from logs_producer import generate_log_lines, make_filename
import os

INPUT_PATH = "/opt/spark/work-dir/data/streaming/logs/"
os.makedirs(INPUT_PATH, exist_ok=True)

for i in range(1, 4):
    content  = generate_log_lines(n_lines=50)
    filename = make_filename(i)
    path     = os.path.join(INPUT_PATH, filename)
    with open(path, "w") as f:
        f.write(content)
    print(f"[{i}/3] Written: {path}")

[1/3] Written: /opt/spark/work-dir/data/streaming/logs/logs_20260408_195928_001.log
[2/3] Written: /opt/spark/work-dir/data/streaming/logs/logs_20260408_195928_002.log
[3/3] Written: /opt/spark/work-dir/data/streaming/logs/logs_20260408_195928_003.log


In [9]:
import pyspark.sql.functions as F
from pathlib import Path
import shutil

logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

logs_df = (
    su.spark.readStream
    .format("text")
    .option("maxFilesPerTrigger", 1)
    .schema(logs_schema)
    .load(INPUT_PATH)
)

In [10]:
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("raw_line"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "raw_line")
    .filter(F.col("timestamp").isNotNull())
    .filter(F.col("level") == "ERROR")
)

In [11]:
summary_df = (
    parsed_df
    .groupBy("server", "level")
    .count()
    .orderBy("server", "level")
)

In [ ]:
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

query_events = (
    parsed_df.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .option("checkpointLocation", checkpoint_path)
    .queryName("parsed_logs")
    .start()
)

query_summary = (
    summary_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .queryName("summary_logs")
    .start()
)

print("Streaming active. Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()

26/04/08 20:00:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/08 20:00:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-95d1fdc8-c85a-4829-9a61-8266a5fdb82b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/08 20:00:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming active. Press Ctrl+C to stop.

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-----+---------------------------+-------------+
|timestamp          |level|message                    |server       |
+-------------------+-----+---------------------------+-------------+
|2026-04-08 19:59:32|ERROR|500 Internal Server Error  |server-node-4|
|2026-04-08 19:59:35|ERROR|Database connection timeout|server-node-2|
|2026-04-08 19:59:43|ERROR|Database connection timeout|server-node-3|
|2026-04-08 19:59:53|ERROR|404 Not Found              |server-node-1|
|2026-04-08 20:00:28|ERROR|Database connection timeout|server-node-1|
|2026-04-08 20:00:55|ERROR|Disk full                  |server-node-2|
|2026-04-08 20:01:34|ERROR|Disk full                  |server-node-4|
|2026-04-08 20:01:50|ERROR|Authentication failed      |server-node-2|
|2026-04-08 20:02:30|ERROR|Disk full                  |server-node-4|
|2026-04-08 20:02:32|E